In [1]:
def generate_Qk_horizon(k):
    """
    生成第 k 级筛法在有效视界 (N < p_{k+1}^2) 内的符号序列
    k=1(筛2), k=2(筛2,3), k=3(筛2,3,5)...
    """
    primes = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53]
    used_primes = primes[:k]
    next_prime = primes[k]
    horizon = next_prime ** 2  # 论文定义的有效视界
    
    seq = ['L'] * horizon
    seq[0] = 'R' # 数字 0 永远被筛除
    
    for p in used_primes:
        for i in range(0, horizon, p):
            seq[i] = 'R'
            
    return "".join(seq), horizon, next_prime

def mss_compare(s1, s2):
    """
    【核心】MSS 奇偶字典序比较 (Parity-Lexicographical Order)
    返回: -1 (s1 < s2), 1 (s1 > s2), 0 (相等)
    """
    for i in range(min(len(s1), len(s2))):
        if s1[i] == s2[i]:
            continue
            
        # 找到分歧点！统计前面的公共前缀中 'R' 的个数
        prefix = s1[:i]
        r_count = prefix.count('R')
        
        # 基础自然顺序判定 (字母表里 L < R)
        if s1[i] == 'L' and s2[i] == 'R':
            base_cmp = -1
        else:
            base_cmp = 1
            
        # 物理动力学法则：奇数个 R 会导致相空间被抛物线折叠，比较法则必须彻底颠倒！
        if r_count % 2 == 1:
            return -base_cmp
        else:
            return base_cmp
            
    return 0 # 前缀完全相同

def check_admissibility(seq):
    """检查最大化条件 (Maximal Condition)"""
    for shift in range(1, len(seq)):
        shifted = seq[shift:]
        orig = seq[:len(shifted)] # 原序列截断到与移位序列等长
        
        # 如果 原序列 < 移位序列，则非法（发生拓扑逃逸）！
        if mss_compare(orig, shifted) == -1:
            return False, shift, orig, shifted
            
    return True, -1, "", ""

# ================= 运行测试 =================
for k in range(1, 6):
    qk_str, horizon, next_p = generate_Qk_horizon(k)
    is_adm, bad_shift, orig, shifted = check_admissibility(qk_str)
    
    print(f"\n=== Q_{k} (视界 N < {next_p}^2 = {horizon}) ===")
    print(f"合法吗? {'✅ 是 (Admissible)' if is_adm else '❌ 否 (Illegal)'}")
    
    if not is_adm:
        print(f"  崩塌发生于偏移量 m = {bad_shift}")
        # 寻找具体分歧点
        for i in range(len(orig)):
            if orig[i] != shifted[i]:
                prefix = orig[:i]
                r_count = prefix.count('R')
                print(f"  --- 案发现场 ---")
                print(f"  引起崩塌的物理数字位置: {bad_shift + i}")
                print(f"  公共前缀: '{prefix}'")
                print(f"  前缀中 'R' 的个数: {r_count} -> 【{'奇数 (法则反转 R<L)' if r_count%2!=0 else '偶数 (自然顺序 L<R)'}】")
                print(f"  原序列字符: '{orig[i]}', 移位序列字符: '{shifted[i]}'")
                print(f"  结论: 试图用 'R' 赢 'L'，但因为空间反转，'R' 反而变小，原序列输了！")
                break


=== Q_1 (视界 N < 3^2 = 9) ===
合法吗? ✅ 是 (Admissible)

=== Q_2 (视界 N < 5^2 = 25) ===
合法吗? ✅ 是 (Admissible)

=== Q_3 (视界 N < 7^2 = 49) ===
合法吗? ❌ 否 (Illegal)
  崩塌发生于偏移量 m = 22
  --- 案发现场 ---
  引起崩塌的物理数字位置: 31
  公共前缀: 'RLRRRRRLR'
  前缀中 'R' 的个数: 7 -> 【奇数 (法则反转 R<L)】
  原序列字符: 'R', 移位序列字符: 'L'
  结论: 试图用 'R' 赢 'L'，但因为空间反转，'R' 反而变小，原序列输了！

=== Q_4 (视界 N < 11^2 = 121) ===
合法吗? ✅ 是 (Admissible)

=== Q_5 (视界 N < 13^2 = 169) ===
合法吗? ❌ 否 (Illegal)
  崩塌发生于偏移量 m = 112
  --- 案发现场 ---
  引起崩塌的物理数字位置: 125
  公共前缀: 'RLRRRRRRRRRRR'
  前缀中 'R' 的个数: 12 -> 【偶数 (自然顺序 L<R)】
  原序列字符: 'L', 移位序列字符: 'R'
  结论: 试图用 'R' 赢 'L'，但因为空间反转，'R' 反而变小，原序列输了！
